# 11 · Diagnostic health features from the raw RPT and EIS files

**Why this notebook exists.** The forecasting reframe (decision D16) needs inputs that carry health information. Notebooks 02 and 10 showed that the handover's cycle-level features do not. The public archive (S1 in `docs/EXTERNAL_DATA.md`) contains what the handover lacks: every reference performance test (RPT) and, from RPT 5 on, EIS spectra.

**Source files.**
- `scripts/fetch_osf_diagnostics.py` downloads the archive and checks its SHA-256.
- `scripts/build_diagnostic_features.py` builds one row of features per (battery, RPT) with `src/diagnostic_features.py`. It logs every file choice and checks the RPT 7 decoding against the authors' own values.

**Features, all measured at the RPT (no protocol current is used, because currents are set from the previous capacity):**

| Group | Features |
|---|---|
| Capacity test (C/20 discharge) | charge in each window between the pulse voltages; peak positions and heights of the incremental-capacity curve −dQ/dV |
| Pulses (C/2, 8 s) | resistance after 0.1, 1 and 8 s at 4.00, 3.63 and 3.26 V |
| EIS (3 voltages) | R0 (Z′ where Z″ = 0), arc resistance, arc peak, Z′ at 1 kHz … 0.01 Hz |

## 1. Load the feature table and the selection log

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

from src import diagnostic_features as F
from src import raw_diagnostics as rd

dp.check_data_files(DATA_ROOT, required=["diagnostic_features.csv", "diagnostic_selection_log.csv", "final_battery_health_dataset.csv"])
feat = dp.read_dataset(DATA_ROOT / "diagnostic_features.csv")
feat["rpt_start"] = pd.to_datetime(feat["rpt_start"])
log = pd.read_csv(DATA_ROOT / "diagnostic_selection_log.csv")
FEATURES = F.feature_columns(feat)
groups = [g for g in dp.BATTERY_GROUPS if g in set(feat[dp.GROUP_COL])]
MARKERS = dict(zip(dp.BATTERY_GROUPS, ["o", "s", "^", "D", "v", "P"]))
print(f"{len(feat)} (battery, RPT) rows, {len(FEATURES)} features")
display(feat[FEATURES].notna().sum().rename("non-missing").to_frame().T)
display(log.groupby("rule").size().rename("files").to_frame())

## 2. Data-quality flags

These are recorded, not repaired; each has a fixed rule in `src/diagnostic_features.py`.
- **RPT 2 at about 20 °C.** The article's Table 3 notes an incorrectly configured chamber setpoint. Capacity and resistance at RPT 2 are not comparable with 25 °C RPTs. This also affects the SOH label at RPT 2.
- **Off-protocol pulses.** RPT 1 pulsed at 0.525 A at 4.00 V, and v4 and w10 pulsed at 1.05 A in RPT 17–19. Their resistances are set to missing.
- **Galvanostatic EIS** (`_gal` files, w8 RPT 17 and 19) is excluded, because every other spectrum is potentiostatic.
- **RPT 7 EIS** comes from the aggregate files. It is validated exactly against the authors' `R_HF`.

In [ ]:
flags = feat.assign(
    temp_c=feat["rpt_temp_mean"].round(1),
    pulses_off_protocol=feat["nonstandard_pulses"].fillna(""),
    eis_voltages=feat[[f"eis_r0_{v}" for v in ("3.26", "3.63", "4.00")]].notna().sum(axis=1),
)
display(flags.pivot(index="rpt_number", columns=dp.GROUP_COL, values="temp_c").round(1))
display(flags[flags["pulses_off_protocol"] != ""][[dp.GROUP_COL, "rpt_number", "pulses_off_protocol"]])
display(flags.pivot(index="rpt_number", columns=dp.GROUP_COL, values="eis_voltages").loc[5:])

## 3. Capacity definitions

The handover's `Capacity_Ah` (the SOH label) is the total discharged charge over the RPT, including the pulses. `q_c20_ah` counts only the C/20 steps, the article's definition. The two should differ by a nearly constant pulse contribution.

In [ ]:
diff = (feat["Capacity_Ah"] - feat["q_c20_ah"]) * 1000
print(f"Capacity_Ah - q_c20_ah: median {diff.median():.1f} mAh, range {diff.min():.1f} to {diff.max():.1f} mAh")
display(feat.assign(diff_mah=diff).groupby("rpt_number")["diff_mah"].agg(["min", "max"]).round(1).T)

## 4. Incremental capacity over ageing

In [ ]:
cache = DATA_ROOT / rd.CACHE_SUBDIR
raw = DATA_ROOT / rd.RAW_SUBDIR
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), sharex=True, sharey=True)
cmap = plt.get_cmap("Blues")
for ax, g in zip(axes.flat, groups):
    rows = feat[feat[dp.GROUP_COL] == g]
    n = rows["rpt_number"].max()
    for _, r in rows.iterrows():
        path = next(raw.rglob(r["rpt_file"]))
        ts = rd.timeseries(rd.cached_sheets(path, cache))
        segs = F._segments(ts)
        c20 = [s for s in segs if F._kind(s) == "c20"]
        ax.plot(F.IC_GRID, F.ic_curve(c20), color=cmap(0.3 + 0.7 * (r["rpt_number"] - 1) / max(n - 1, 1)), linewidth=1.2)
    ax.set_title(g, loc="left")
    ax.set_xlim(3.3, 4.15)
for ax in axes[-1]:
    ax.set_xlabel("Voltage (V)")
for ax in axes[:, 0]:
    ax.set_ylabel("−dQ/dV (Ah/V)")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(1, 19))
fig.colorbar(sm, ax=axes, shrink=0.8, label="RPT number (light = early)")
fig.suptitle("Incremental capacity of the C/20 discharge, all RPTs", x=0.02, ha="left")
savefig(fig, "11_ic_curves")
plt.show()

## 5. EIS over ageing (3.63 V)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), sharex=True, sharey=True)
for ax, g in zip(axes.flat, groups):
    rows = feat[(feat[dp.GROUP_COL] == g) & feat["eis_file_3.63"].notna()]
    for _, r in rows.iterrows():
        f = r["eis_file_3.63"]
        if f.endswith("_ALL.xlsx"):
            z = F.read_aggregate_eis(next(raw.rglob(f)), ref_freq)[g]
        else:
            z = rd.impedance(rd.cached_sheets(next(raw.rglob(f)), cache))
            ref_freq = z["freq_hz"].to_numpy()
        ax.plot(z["z_real"] * 1e3, -z["z_imag"] * 1e3, color=cmap(0.3 + 0.7 * (r["rpt_number"] - 5) / 14), linewidth=1.2)
    ax.set_title(g, loc="left")
    ax.set_xlim(23, 36); ax.set_ylim(-1, 6)
for ax in axes[-1]:
    ax.set_xlabel("Z′ (mΩ)")
for ax in axes[:, 0]:
    ax.set_ylabel("−Z″ (mΩ)")
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(5, 19)), ax=axes, shrink=0.8, label="RPT number (light = early)")
fig.suptitle("Nyquist plots at 3.63 V, RPT 5–19", x=0.02, ha="left")
savefig(fig, "11_nyquist_363")
plt.show()

## 6. Feature trajectories

A few representative features against time since the first RPT, one line per battery. RPT 2 (20 °C) is drawn with an open marker.

In [ ]:
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
first = health.groupby(dp.GROUP_COL)["Test_Date"].min()
feat["days"] = (feat["Test_Date"] - feat[dp.GROUP_COL].map(first)).dt.days
show = {"SOH_Percent": "SOH (%)", "r_pulse_3.63_1s": "Pulse R, 3.63 V, 1 s (mΩ)",
        "eis_r0_3.63": "EIS R0, 3.63 V (mΩ)", "eis_rarc_3.26": "EIS arc R, 3.26 V (mΩ)",
        "ic_height_p5": "IC peak height near 4.07 V (Ah/V)", "q_3.26_2.50": "Charge 3.26→2.50 V (Ah)"}
scale = {k: (1e3 if k.startswith(("r_pulse", "eis_")) else 1) for k in show}
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), sharex=True)
for ax, (col, label) in zip(axes.flat, show.items()):
    for g in groups:
        s = feat[feat[dp.GROUP_COL] == g].sort_values("days")
        ax.plot(s["days"], s[col] * scale[col], marker=MARKERS[g], markersize=4, color=GROUP_COLORS[g], label=g, linewidth=1.4)
        r2 = s[s["rpt_number"] == 2]
        ax.plot(r2["days"], r2[col] * scale[col], marker=MARKERS[g], markersize=7, markerfacecolor="white", color=GROUP_COLORS[g], linestyle="none")
    ax.set_title(label, loc="left", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("Days since first RPT")
axes[0, 0].legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
savefig(fig, "11_feature_trajectories")
plt.show()

## 7. Association with SOH and with the next SOH change

Within each battery, Spearman ρ of each feature at RPT *k* with:
- the SOH at RPT *k* (does the feature track health?);
- the SOH change to the next RPT, ΔSOH = SOH(*k*+1) − SOH(*k*). This is the quantity a forecaster must predict beyond persistence.

RPT 2 (20 °C) is excluded, as are pairs spanning the long rest before RPT 16, where SOH rises after rest (`docs/DATA_REPORT.md`). With about 15 points per battery, only consistent signs across batteries are meaningful.

In [ ]:
d = feat.sort_values([dp.GROUP_COL, "rpt_number"]).copy()
d["dsoh_next"] = d.groupby(dp.GROUP_COL)["SOH_Percent"].shift(-1) - d["SOH_Percent"]
d.loc[d["rpt_number"].isin([15]), "dsoh_next"] = np.nan         # pair spanning the rest before RPT 16
d = d[d["rpt_number"] != 2]
cands = ["days"] + FEATURES


def within(target):
    return pd.DataFrame({g: s[cands + [target]].corr(method="spearman")[target].drop(target)
                         for g, s in d.groupby(dp.GROUP_COL)}).reindex(columns=groups)


rho_soh, rho_dsoh = within("SOH_Percent"), within("dsoh_next")
summary = pd.DataFrame({
    "median rho vs SOH": rho_soh.median(axis=1),
    "same sign (of 6), SOH": (np.sign(rho_soh).eq(np.sign(rho_soh.median(axis=1)), axis=0)).sum(axis=1),
    "median rho vs next dSOH": rho_dsoh.median(axis=1),
    "same sign (of 6), dSOH": (np.sign(rho_dsoh).eq(np.sign(rho_dsoh.median(axis=1)), axis=0)).sum(axis=1),
}).sort_values("median rho vs SOH", key=abs, ascending=False)
display(summary.round(2))

top = summary.index[:20]
fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
for ax, rho, title in ((axes[0], rho_soh, "vs SOH at the same RPT"), (axes[1], rho_dsoh, "vs SOH change to the next RPT")):
    m = rho.loc[top].to_numpy(dtype=float)
    im = ax.imshow(m, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(groups)), groups)
    ax.set_title(title, loc="left")
    ax.grid(False)
    for (i, j), v in np.ndenumerate(m):
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(v) > 0.6 else "#1a1a19")
axes[0].set_yticks(range(len(top)), top, fontsize=8)
fig.colorbar(im, ax=axes, shrink=0.6, label="Within-battery Spearman ρ")
savefig(fig, "11_feature_association")
plt.show()

## 8. Save results

In [ ]:
import json
out = RESULTS_DIR / "diagnostic_features.json"
payload = {
    "environment": dp.environment_info(REPO_ROOT),
    "n_rows": int(len(feat)), "n_features": len(FEATURES),
    "non_missing": feat[FEATURES].notna().sum().astype(int).to_dict(),
    "selection_rules": log.groupby("rule").size().astype(int).to_dict(),
    "capacity_minus_c20_mah": {"median": float(diff.median()), "min": float(diff.min()), "max": float(diff.max())},
    "within_battery_spearman_summary": summary.reset_index(names="feature").to_dict(orient="records"),
}
out.write_text(json.dumps(payload, indent=2, default=str))
print("wrote", out.relative_to(REPO_ROOT))

## 9. Interpretation (fill in from the executed run)

- **Validation.** RPT 7 R0 equals the authors' `R_HF` exactly, and all handover capacities are reproduced (notebook 01 and `scripts/verify_external_data.py`).
- **Trends.** Resistance features (pulse and EIS) rise, and IC peak heights fall, as SOH falls. The same trend-with-time caveat as notebook 10 applies: a feature that drifts monotonically will correlate with SOH.
- **Forecasting relevance.** Section 7's ΔSOH column is the useful test. A feature with a consistent sign across batteries against the *next* change carries information beyond persistence. If none does, forecasting will rest on the SOH history and usage, and the paper must say so.